In [7]:
import pandas as pd

# Load the cleaned dataset from data preparation phase
input_path = '../data/processed/final_phishing_dataset.csv'
df = pd.read_csv(input_path)
print(f"Loaded {len(df)} URLs")

# Extract lexical features - these capture structural patterns in URLs
# Phishing URLs tend to be longer, have more special characters, and contain suspicious keywords
df['url_len'] = df['url'].apply(lambda x: len(str(x)))
df['n_dots'] = df['url'].apply(lambda x: str(x).count('.'))
df['n_hyphens'] = df['url'].apply(lambda x: str(x).count('-'))
df['n_digits'] = df['url'].apply(lambda x: sum(c.isdigit() for c in str(x)))
df['n_slashes'] = df['url'].apply(lambda x: str(x).count('/'))

# Check for words commonly used in phishing URLs to create urgency or mimic legitimate sites
sus_keywords = ['login', 'secure', 'account', 'update', 'banking', 'verify']
df['has_sus_kw'] = df['url'].apply(lambda x: 1 if any(w in str(x).lower() for w in sus_keywords) else 0)

# HTTPS presence - legitimate sites typically use HTTPS, but phishers increasingly do too
df['has_https'] = df['url'].apply(lambda x: 1 if str(x).lower().startswith('https://') else 0)

output_path = '../data/processed/phishing_data_with_features.csv'
df.to_csv(output_path, index=False)

print(f"Features: url_len, n_dots, n_hyphens, n_digits, n_slashes, has_sus_kw, has_https")
print(f"Saved to: {output_path}")
print(df.head())

Loaded 39999 URLs
Features: url_len, n_dots, n_hyphens, n_digits, n_slashes, has_sus_kw, has_https
Saved to: ../data/processed/phishing_data_with_features.csv
                                           url  label  url_len  n_dots  \
0      https://jetstar.com/article?ref=default      0       39       1   
1   https://eu.jotform.com/app/250522810922348      1       42       2   
2  https://www.doujindesu.tv/user?profile=home      0       43       2   
3      https://malrkoumdtrtya.firebaseapp.com/      1       39       2   
4         https://www.signupgenius.com/careers      0       36       2   

   n_hyphens  n_digits  n_slashes  has_sus_kw  has_https  
0          0         0          3           0          1  
1          0        15          4           0          1  
2          0         0          3           0          1  
3          0         0          3           0          1  
4          0         0          3           0          1  


In [8]:
# Validate that features show meaningful differences between classes
# If phishing URLs have higher averages for these features, they'll be useful for classification
validation_table = df.groupby('label')[['url_len', 'n_dots', 'n_hyphens', 'n_digits', 'has_sus_kw']].mean()
print("Feature averages by class (0=Legit, 1=Phishing):")
print(validation_table)

Feature averages by class (0=Legit, 1=Phishing):
         url_len   n_dots  n_hyphens  n_digits  has_sus_kw
label                                                     
0      28.239350  1.38100    0.12945  0.213300    0.021350
1      62.538127  2.00855    0.80764  6.653933    0.036802


In [9]:
import sys
import os
import pandas as pd

# Add project root to path so we can import the automata module
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

# Import the automata-based feature extractor (rule-based pattern matching)
try:
    from src.automata.automata_features import extract_features_url
    print("Imported automata features successfully")
except ImportError as e:
    print(f"Import Error: {e}")

def get_automata_vector(url):
    """Wrapper to extract automata features and return as a pandas Series."""
    try:
        features, _ = extract_features_url(str(url))
        return pd.Series(features)
    except Exception as e:
        if "printed_error" not in get_automata_vector.__dict__:
            print(f"Error: {e}")
            get_automata_vector.printed_error = True
        return pd.Series([0]*20)

# Load the dataset with lexical features
df = pd.read_csv('../data/processed/phishing_data_with_features.csv')

# Apply automata rules to each URL - these detect specific phishing patterns
# using finite state machines (e.g., IP addresses in URLs, suspicious TLDs)
print("Running automata rules...")
automata_cols = df['url'].apply(get_automata_vector)

# Create hybrid dataset combining lexical + automata features
hybrid_df = pd.concat([df, automata_cols], axis=1)
output_path = '../data/processed/hybrid_dataset.csv'
hybrid_df.to_csv(output_path, index=False)

print(f"Done! Saved to: {output_path}")
print(f"Shape: {hybrid_df.shape}")
print(hybrid_df.iloc[:, -5:].head())

Imported automata features successfully
Running automata rules...
Done! Saved to: ../data/processed/hybrid_dataset.csv
Shape: (39999, 29)
   match_url_16  match_url_17  match_url_18  match_url_19  match_url_20
0             0             0             0             0             0
1             0             0             0             0             0
2             0             0             0             0             0
3             0             0             0             0             0
4             0             0             0             0             0
